# Prepping the Pod (Before you get to Jupyter)

Following this:

https://docs.runpod.io/tutorials/pods/run-ollama

1. Log in to your RunPod account and choose + GPU Pod. <br>
2. Choose a GPU Pod like A40.<br>
3. From the availble templates, select the lastet PyTorch template.<br>
4. Select Customize Deployment.<br>
- Add the port 11434 to the list of exposed ports. This port is used by Ollama for HTTP API requests.<br>
- Add the following environment variable to your Pod to allow Ollama to bind to the HTTP port:<br>
- Key: OLLAMA_HOST<br>
- Value: 0.0.0.0<br>
5. Select Set Overrides, Continue, then Deploy.<br>

It is best to put both the container and the volume at whatever size you need. I was running into issues. Just do both to avoid any...

**If you have to adjust the sizes at all, you will lose your downloaded models, the environment setup, and have to repeat the process**

Can run the following in both ssh or Jupyter, but let's stick to SSH then move over to jupyter later.

**apt update**<br>
**apt install lshw -y**

In [ ]:
# !apt update
# !apt install lshw -y

Method 1 (preferred so I can watch the post/gets in the SSH terminal):

I had this method working for sure in the SSH terminal:
- **curl https://ollama.ai/install.sh | sh**
- **ollama serve**

  
[not working how I need it to] Method 2 (from the guide):

(curl -fsSL https://ollama.com/install.sh | sh && ollama serve > ollama.log 2>&1) &

In [17]:
# !curl https://ollama.ai/install.sh | sh

# Ollama Serve MUST BE RAN IN THE SSH CONSOLE, NOT HERE IN THE NOTEBOOK!

Go run "Ollama serve" in the ssh

In [9]:

# # instead i used....
# # spinning up the server
# import subprocess

# # Start the ollama server
# ollama_process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# # Add a note to remind yourself
# print("Ollama server started in the background.")


# Start Jupyter Portion - Installs.
For some reason there's an issue with 0.4.0... install the older versions.

In [5]:
!nvidia-smi

Sun Dec 29 16:23:41 2024       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.127.08             Driver Version: 550.127.08     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A40                     On  |   00000000:D2:00.0 Off |                    0 |
|  0%   34C    P8             21W /  300W |       4MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
!pip install ollama==0.4.0 lm_eval lm_eval[api] -q


[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python -m pip install --upgrade pip


In [7]:
!pip install ollama==0.3.3

  Attempting uninstall: ollama
    Found existing installation: ollama 0.4.0
    Uninstalling ollama-0.4.0:
      Successfully uninstalled ollama-0.4.0

[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python -m pip install --upgrade pip


# Pulling models to run

## Manual model input arrays

In [2]:
# Define the list of models to pull
# Test List A40
model_list = [
    "hf.co/bartowski/Llama-3.2-3B-Instruct-GGUF:Q3_K_L",
    "hf.co/bartowski/Llama-3.2-3B-Instruct-GGUF:Q4_K_S",
    "hf.co/bartowski/Phi-3.5-mini-instruct-GGUF:Q4_K_L", # 2.47GB
    "hf.co/bartowski/Phi-3.5-mini-instruct-GGUF:Q2_K_L", # 1.51GB
]
# Test List 2 A40
model_list = [
"hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q8_0", # 8.54 GB
"hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q6_K_L", # 6.85 GB
"hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q6_K", # 6.60 GB
"hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q5_K_L", # 6.06 GB
]

# Batch 1 A40
model_list = [
"hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:F32", # 32.13 GB
"hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q8_0", # 8.54 GB
"hf.co/bartoswski/Meta-Llama-3.1-8B-Instruct-GGUF:Q6_K_L", # 6.85 GB
"hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q6_K", # 6.60 GB
"hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q5_K_L", # 6.06 GB
"hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q5_K_M", # 5.73 GB
"hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q5_K_S", # 5.60 GB
"hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q4_K_L", # 5.31 GB
"hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q4_K_M", # 4.92 GB
"hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q3_K_XL", # 4.78 GB
"hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q4_K_S", # 4.69 GB
"hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:IQ4_NL", # 4.68 GB
"hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q4_0_8_8", # 4.66 GB
"hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q4_0_4_8", # 4.66 GB
"hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q4_0_4_4", # 4.66 GB
"hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:IQ4_XS", # 4.45 GB
"hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q3_K_L", # 4.32 GB
"hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q3_K_M", # 4.02 GB
"hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:IQ3_M", # 3.78 GB
"hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q2_K_L", # 3.69 GB
"hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q3_K_S", # 3.66 GB
"hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:IQ3_XS", # 3.52 GB
"hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q2_K", # 3.18 GB
"hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:IQ2_M", # 2.95 GB
]

In [10]:
# Batch 2 A40 -- 150GB+, use 250GB
model_list = [
"hf.co/bartowski/Phi-3.5-mini-instruct-GGUF:F32", # 15.29 GB
"hf.co/bartowski/Phi-3.5-mini-instruct-GGUF:Q8_0", # 4.06 GB
"hf.co/bartowski/Phi-3.5-mini-instruct-GGUF:Q6_K_L", # 3.18 GB
"hf.co/bartowski/Phi-3.5-mini-instruct-GGUF:Q6_K", # 3.14 GB
"hf.co/bartowski/Phi-3.5-mini-instruct-GGUF:Q5_K_L", # 2.88 GB
"hf.co/bartowski/Phi-3.5-mini-instruct-GGUF:Q5_K_M", # 2.82 GB
"hf.co/bartowski/Phi-3.5-mini-instruct-GGUF:Q5_K_S", # 2.64 GB
"hf.co/bartowski/Phi-3.5-mini-instruct-GGUF:Q4_K_L", # 2.47 GB
"hf.co/bartowski/Phi-3.5-mini-instruct-GGUF:Q4_K_M", # 2.39 GB
"hf.co/bartowski/Phi-3.5-mini-instruct-GGUF:Q4_K_S", # 2.19 GB
"hf.co/bartowski/Phi-3.5-mini-instruct-GGUF:Q4_0_8_8", # 2.18 GB
"hf.co/bartowski/Phi-3.5-mini-instruct-GGUF:Q4_0_4_8", # 2.18 GB
"hf.co/bartowski/Phi-3.5-mini-instruct-GGUF:Q4_0_4_4", # 2.18 GB
"hf.co/bartowski/Phi-3.5-mini-instruct-GGUF:Q4_0", # 2.18 GB
"hf.co/bartowski/Phi-3.5-mini-instruct-GGUF:Q3_K_XL", # 2.17 GB
"hf.co/bartowski/Phi-3.5-mini-instruct-GGUF:Q3_K_L", # 2.09 GB
"hf.co/bartowski/Phi-3.5-mini-instruct-GGUF:IQ4_XS", # 2.06 GB
"hf.co/bartowski/Phi-3.5-mini-instruct-GGUF:Q3_K_M", # 1.96 GB
"hf.co/bartowski/Phi-3.5-mini-instruct-GGUF:IQ3_M", # 1.86 GB
"hf.co/bartowski/Phi-3.5-mini-instruct-GGUF:Q3_K_S", # 1.68 GB
"hf.co/bartowski/Phi-3.5-mini-instruct-GGUF:IQ3_XS", # 1.63 GB
"hf.co/bartowski/Phi-3.5-mini-instruct-GGUF:Q2_K_L", # 1.51 GB
"hf.co/bartowski/Phi-3.5-mini-instruct-GGUF:Q2_K", # 1.42 GB
"hf.co/bartowski/Phi-3.5-mini-instruct-GGUF:IQ2_M", # 1.32 GB

"hf.co/bartowski/Mistral-7B-Instruct-v0.3-GGUF:Q8_0", # 7.70 GB
"hf.co/bartowski/Mistral-7B-Instruct-v0.3-GGUF:Q6_K", # 5.94 GB
"hf.co/bartowski/Mistral-7B-Instruct-v0.3-GGUF:Q5_K_M", # 5.13 GB
"hf.co/bartowski/Mistral-7B-Instruct-v0.3-GGUF:Q5_K_S", # 5.00 GB
"hf.co/bartowski/Mistral-7B-Instruct-v0.3-GGUF:Q4_K_M", # 4.37 GB
"hf.co/bartowski/Mistral-7B-Instruct-v0.3-GGUF:Q4_K_S", # 4.14 GB
"hf.co/bartowski/Mistral-7B-Instruct-v0.3-GGUF:IQ4_NL", # 4.13 GB
"hf.co/bartowski/Mistral-7B-Instruct-v0.3-GGUF:IQ4_XS", # 3.91 GB
"hf.co/bartowski/Mistral-7B-Instruct-v0.3-GGUF:Q3_K_L", # 3.82 GB
"hf.co/bartowski/Mistral-7B-Instruct-v0.3-GGUF:Q3_K_M", # 3.52 GB
"hf.co/bartowski/Mistral-7B-Instruct-v0.3-GGUF:IQ3_M", # 3.28 GB
"hf.co/bartowski/Mistral-7B-Instruct-v0.3-GGUF:IQ3_S", # 3.18 GB
"hf.co/bartowski/Mistral-7B-Instruct-v0.3-GGUF:Q3_K_S", # 3.16 GB
"hf.co/bartowski/Mistral-7B-Instruct-v0.3-GGUF:IQ3_XS", # 3.02 GB
"hf.co/bartowski/Mistral-7B-Instruct-v0.3-GGUF:IQ3_XXS", # 2.83 GB
"hf.co/bartowski/Mistral-7B-Instruct-v0.3-GGUF:Q2_K", # 2.72 GB
"hf.co/bartowski/Mistral-7B-Instruct-v0.3-GGUF:IQ2_M", # 2.50 GB
"hf.co/bartowski/Mistral-7B-Instruct-v0.3-GGUF:IQ2_S", # 2.31 GB
"hf.co/bartowski/Mistral-7B-Instruct-v0.3-GGUF:IQ2_XS", # 2.20 GB
"hf.co/bartowski/Mistral-7B-Instruct-v0.3-GGUF:IQ2_XXS", # 1.99 GB
"hf.co/bartowski/Mistral-7B-Instruct-v0.3-GGUF:IQ1_M", # 1.75 GB
"hf.co/bartowski/Mistral-7B-Instruct-v0.3-GGUF:IQ1_S", # 1.61 GB
]

In [ ]:
## Batch 3 - A40

In [18]:
# Batch 3 A40 280GB, use 400GB
model_list = [
"hf.co/bartowski/falcon-11B-GGUF:Q8_0", # 11.80 GB
"hf.co/bartowski/falcon-11B-GGUF:Q6_K", # 9.17 GB
"hf.co/bartowski/falcon-11B-GGUF:Q5_K_M", # 8.20 GB
"hf.co/bartowski/falcon-11B-GGUF:Q5_K_S", # 7.73 GB
"hf.co/bartowski/falcon-11B-GGUF:Q4_K_M", # 6.84 GB
"hf.co/bartowski/falcon-11B-GGUF:Q4_K_S", # 6.38 GB
"hf.co/bartowski/falcon-11B-GGUF:IQ4_NL", # 6.38 GB
"hf.co/bartowski/falcon-11B-GGUF:IQ4_XS", # 6.04 GB
"hf.co/bartowski/falcon-11B-GGUF:Q3_K_L", # 5.81 GB
"hf.co/bartowski/falcon-11B-GGUF:Q3_K_M", # 5.43 GB
"hf.co/bartowski/falcon-11B-GGUF:IQ3_M", # 5.20 GB
"hf.co/bartowski/falcon-11B-GGUF:IQ3_S", # 4.94 GB
"hf.co/bartowski/falcon-11B-GGUF:Q3_K_S", # 4.94 GB
"hf.co/bartowski/falcon-11B-GGUF:IQ3_XS", # 4.80 GB
"hf.co/bartowski/falcon-11B-GGUF:IQ3_XXS", # 4.44 GB
"hf.co/bartowski/falcon-11B-GGUF:Q2_K", # 4.25 GB
"hf.co/bartowski/falcon-11B-GGUF:IQ2_M", # 3.94 GB
"hf.co/bartowski/falcon-11B-GGUF:IQ2_S", # 3.66 GB
"hf.co/bartowski/falcon-11B-GGUF:IQ2_XS", # 3.44 GB
"hf.co/bartowski/falcon-11B-GGUF:IQ2_XXS", # 3.13 GB
"hf.co/bartowski/falcon-11B-GGUF:IQ1_M", # 2.77 GB
"hf.co/bartowski/falcon-11B-GGUF:IQ1_S", # 2.56 GB

"hf.co/bartowski/Phi-3-medium-4k-instruct-GGUF:Q8_0", # 14.83 GB
"hf.co/bartowski/Phi-3-medium-4k-instruct-GGUF:Q6_K_L", # 11.53 GB
"hf.co/bartowski/Phi-3-medium-4k-instruct-GGUF:Q6_K", # 11.45 GB
"hf.co/bartowski/Phi-3-medium-4k-instruct-GGUF:Q5_K_L", # 10.18 GB
"hf.co/bartowski/Phi-3-medium-4k-instruct-GGUF:Q5_K_M", # 10.07 GB
"hf.co/bartowski/Phi-3-medium-4k-instruct-GGUF:Q5_K_S", # 9.62 GB
"hf.co/bartowski/Phi-3-medium-4k-instruct-GGUF:Q4_K_L", # 8.69 GB
"hf.co/bartowski/Phi-3-medium-4k-instruct-GGUF:Q4_K_M", # 8.57 GB
"hf.co/bartowski/Phi-3-medium-4k-instruct-GGUF:Q4_K_S", # 7.95 GB
"hf.co/bartowski/Phi-3-medium-4k-instruct-GGUF:Q3_K_XL", # 7.63 GB
"hf.co/bartowski/Phi-3-medium-4k-instruct-GGUF:Q3_K_L", # 7.49 GB
"hf.co/bartowski/Phi-3-medium-4k-instruct-GGUF:IQ4_XS", # 7.47 GB
"hf.co/bartowski/Phi-3-medium-4k-instruct-GGUF:Q3_K_M", # 6.92 GB
"hf.co/bartowski/Phi-3-medium-4k-instruct-GGUF:IQ3_M", # 6.47 GB
"hf.co/bartowski/Phi-3-medium-4k-instruct-GGUF:Q3_K_S", # 6.06 GB
"hf.co/bartowski/Phi-3-medium-4k-instruct-GGUF:IQ3_XS", # 5.81 GB
"hf.co/bartowski/Phi-3-medium-4k-instruct-GGUF:Q2_K_L", # 5.30 GB
"hf.co/bartowski/Phi-3-medium-4k-instruct-GGUF:Q2_K", # 5.14 GB
"hf.co/bartowski/Phi-3-medium-4k-instruct-GGUF:IQ2_M", # 4.72 GB

]


In [3]:
# batch 4 - 1xA40, 800, prefer 1000GB to be safe
model_list = [
"hf.co/bartowski/Qwen2.5-32B-Instruct-GGUF:Q8_0", # 34.82 GB
"hf.co/bartowski/Qwen2.5-32B-Instruct-GGUF:Q6_K_L", # 27.26 GB
"hf.co/bartowski/Qwen2.5-32B-Instruct-GGUF:Q6_K", # 26.89 GB
"hf.co/bartowski/Qwen2.5-32B-Instruct-GGUF:Q5_K_L", # 23.74 GB
"hf.co/bartowski/Qwen2.5-32B-Instruct-GGUF:Q5_K_M", # 23.26 GB
"hf.co/bartowski/Qwen2.5-32B-Instruct-GGUF:Q5_K_S", # 22.64 GB
"hf.co/bartowski/Qwen2.5-32B-Instruct-GGUF:Q4_K_L", # 20.43 GB
"hf.co/bartowski/Qwen2.5-32B-Instruct-GGUF:Q4_K_M", # 19.85 GB
"hf.co/bartowski/Qwen2.5-32B-Instruct-GGUF:Q4_K_S", # 18.78 GB
"hf.co/bartowski/Qwen2.5-32B-Instruct-GGUF:Q4_0", # 18.71 GB
"hf.co/bartowski/Qwen2.5-32B-Instruct-GGUF:Q4_0_8_8", # 18.64 GB
"hf.co/bartowski/Qwen2.5-32B-Instruct-GGUF:Q4_0_4_8", # 18.64 GB
"hf.co/bartowski/Qwen2.5-32B-Instruct-GGUF:Q4_0_4_4", # 18.64 GB
"hf.co/bartowski/Qwen2.5-32B-Instruct-GGUF:Q3_K_XL", # 17.93 GB
"hf.co/bartowski/Qwen2.5-32B-Instruct-GGUF:IQ4_XS", # 17.69 GB
"hf.co/bartowski/Qwen2.5-32B-Instruct-GGUF:Q3_K_L", # 17.25 GB
"hf.co/bartowski/Qwen2.5-32B-Instruct-GGUF:Q3_K_M", # 15.94 GB
"hf.co/bartowski/Qwen2.5-32B-Instruct-GGUF:IQ3_M", # 14.81 GB
"hf.co/bartowski/Qwen2.5-32B-Instruct-GGUF:Q3_K_S", # 14.39 GB
"hf.co/bartowski/Qwen2.5-32B-Instruct-GGUF:IQ3_XS", # 13.71 GB
"hf.co/bartowski/Qwen2.5-32B-Instruct-GGUF:Q2_K_L", # 13.07 GB
"hf.co/bartowski/Qwen2.5-32B-Instruct-GGUF:Q2_K", # 12.31 GB
"hf.co/bartowski/Qwen2.5-32B-Instruct-GGUF:IQ2_M", # 11.26 GB
"hf.co/bartowski/Qwen2.5-32B-Instruct-GGUF:IQ2_S", # 10.39 GB
"hf.co/bartowski/Qwen2.5-32B-Instruct-GGUF:IQ2_XS", # 9.96 GB
"hf.co/bartowski/Qwen2.5-32B-Instruct-GGUF:IQ2_XXS", # 9.03 GB

"hf.co/bartowski/gemma-2-27b-it-GGUF:Q8_0", # 28.94 GB
"hf.co/bartowski/gemma-2-27b-it-GGUF:Q6_K_L", # 22.63 GB
"hf.co/bartowski/gemma-2-27b-it-GGUF:Q6_K", # 22.34 GB
"hf.co/bartowski/gemma-2-27b-it-GGUF:Q5_K_L", # 19.69 GB
"hf.co/bartowski/gemma-2-27b-it-GGUF:Q5_K_M", # 19.41 GB
"hf.co/bartowski/gemma-2-27b-it-GGUF:Q5_K_S", # 18.88 GB
"hf.co/bartowski/gemma-2-27b-it-GGUF:Q4_K_L", # 16.93 GB
"hf.co/bartowski/gemma-2-27b-it-GGUF:Q4_K_M", # 16.65 GB
"hf.co/bartowski/gemma-2-27b-it-GGUF:Q4_K_S", # 15.74 GB
"hf.co/bartowski/gemma-2-27b-it-GGUF:IQ4_XS", # 14.81 GB
"hf.co/bartowski/gemma-2-27b-it-GGUF:Q3_K_XL", # 14.81 GB
"hf.co/bartowski/gemma-2-27b-it-GGUF:Q3_K_L", # 14.52 GB
"hf.co/bartowski/gemma-2-27b-it-GGUF:Q3_K_M", # 13.42 GB
"hf.co/bartowski/gemma-2-27b-it-GGUF:IQ3_M", # 12.45 GB
"hf.co/bartowski/gemma-2-27b-it-GGUF:Q3_K_S", # 12.17 GB
"hf.co/bartowski/gemma-2-27b-it-GGUF:IQ3_XS", # 11.55 GB
"hf.co/bartowski/gemma-2-27b-it-GGUF:IQ3_XXS", # 10.75 GB
"hf.co/bartowski/gemma-2-27b-it-GGUF:Q2_K_L", # 10.74 GB
"hf.co/bartowski/gemma-2-27b-it-GGUF:Q2_K", # 10.45 GB
"hf.co/bartowski/gemma-2-27b-it-GGUF:IQ2_M", # 9.40 GB

]

In [ ]:
# batch 5 1xA40
model_list = [
"hf.co/bartowski/Meta-Llama-3.1-70B-Instruct-GGUF:Q3_K_XL", # 38.06 GB
"hf.co/bartowski/Meta-Llama-3.1-70B-Instruct-GGUF:IQ4_XS", # 37.90 GB
"hf.co/bartowski/Meta-Llama-3.1-70B-Instruct-GGUF:Q3_K_L", # 37.14 GB
"hf.co/bartowski/Meta-Llama-3.1-70B-Instruct-GGUF:Q3_K_M", # 34.27 GB
"hf.co/bartowski/Meta-Llama-3.1-70B-Instruct-GGUF:IQ3_M", # 31.94 GB
"hf.co/bartowski/Meta-Llama-3.1-70B-Instruct-GGUF:Q3_K_S", # 30.91 GB
"hf.co/bartowski/Meta-Llama-3.1-70B-Instruct-GGUF:IQ3_XS", # 29.31 GB
"hf.co/bartowski/Meta-Llama-3.1-70B-Instruct-GGUF:Q2_K_L", # 27.40 GB
"hf.co/bartowski/Meta-Llama-3.1-70B-Instruct-GGUF:Q2_K", # 26.38 GB
"hf.co/bartowski/Meta-Llama-3.1-70B-Instruct-GGUF:IQ2_M", # 24.12 GB
"hf.co/bartowski/Meta-Llama-3.1-70B-Instruct-GGUF:IQ2_S", # 22.24 GB
"hf.co/bartowski/Meta-Llama-3.1-70B-Instruct-GGUF:IQ2_XS", # 21.14 GB
"hf.co/bartowski/Meta-Llama-3.1-70B-Instruct-GGUF:IQ2_XXS", # 19.10 GB
"hf.co/bartowski/Meta-Llama-3.1-70B-Instruct-GGUF:IQ1_M", # 16.75 GB

"hf.co/bartowski/Llama-3.3-70B-Instruct-GGUF:Q3_K_M", # 34.27 GB
"hf.co/bartowski/Llama-3.3-70B-Instruct-GGUF:IQ3_M", # 31.94 GB
"hf.co/bartowski/Llama-3.3-70B-Instruct-GGUF:Q3_K_S", # 30.91 GB
"hf.co/bartowski/Llama-3.3-70B-Instruct-GGUF:IQ3_XS", # 29.31 GB
"hf.co/bartowski/Llama-3.3-70B-Instruct-GGUF:IQ3_XXS", # 27.47 GB
"hf.co/bartowski/Llama-3.3-70B-Instruct-GGUF:Q2_K_L", # 27.40 GB
"hf.co/bartowski/Llama-3.3-70B-Instruct-GGUF:Q2_K", # 26.38 GB
"hf.co/bartowski/Llama-3.3-70B-Instruct-GGUF:IQ2_M", # 24.12 GB
"hf.co/bartowski/Llama-3.3-70B-Instruct-GGUF:IQ2_S", # 22.24 GB
"hf.co/bartowski/Llama-3.3-70B-Instruct-GGUF:IQ2_XS", # 21.14 GB
"hf.co/bartowski/Llama-3.3-70B-Instruct-GGUF:IQ2_XXS", # 19.10 GB
"hf.co/bartowski/Llama-3.3-70B-Instruct-GGUF:IQ1_M", # 16.75 GB
 
]

In [6]:
# Batch 6 2xA40 1200GB
model_list = [
"hf.co/bartowski/Meta-Llama-3.1-70B-Instruct-GGUF:Q8_0", # 74.98 GB
"hf.co/bartowski/Meta-Llama-3.1-70B-Instruct-GGUF:Q6_K_L", # 58.40 GB
"hf.co/bartowski/Meta-Llama-3.1-70B-Instruct-GGUF:Q6_K", # 57.89 GB
"hf.co/bartowski/Meta-Llama-3.1-70B-Instruct-GGUF:Q5_K_L", # 50.60 GB
"hf.co/bartowski/Meta-Llama-3.1-70B-Instruct-GGUF:Q5_K_M", # 49.95 GB
"hf.co/bartowski/Meta-Llama-3.1-70B-Instruct-GGUF:Q5_K_S", # 48.66 GB
"hf.co/bartowski/Meta-Llama-3.1-70B-Instruct-GGUF:Q4_K_L", # 43.30 GB
"hf.co/bartowski/Meta-Llama-3.1-70B-Instruct-GGUF:Q4_K_M", # 42.52 GB
"hf.co/bartowski/Meta-Llama-3.1-70B-Instruct-GGUF:Q4_K_S", # 40.35 GB

"hf.co/bartowski/Llama-3.3-70B-Instruct-GGUF:Q4_K_L", # 43.30 GB
"hf.co/bartowski/Llama-3.3-70B-Instruct-GGUF:Q4_K_M", # 42.52 GB
"hf.co/bartowski/Llama-3.3-70B-Instruct-GGUF:Q4_K_S", # 40.35 GB
"hf.co/bartowski/Llama-3.3-70B-Instruct-GGUF:Q4_0", # 40.12 GB
"hf.co/bartowski/Llama-3.3-70B-Instruct-GGUF:IQ4_NL", # 40.05 GB
"hf.co/bartowski/Llama-3.3-70B-Instruct-GGUF:Q4_0_8_8", # 39.97 GB
"hf.co/bartowski/Llama-3.3-70B-Instruct-GGUF:Q4_0_4_8", # 39.97 GB
"hf.co/bartowski/Llama-3.3-70B-Instruct-GGUF:Q4_0_4_4", # 39.97 GB
"hf.co/bartowski/Llama-3.3-70B-Instruct-GGUF:Q3_K_XL", # 38.06 GB
"hf.co/bartowski/Llama-3.3-70B-Instruct-GGUF:IQ4_XS", # 37.90 GB
"hf.co/bartowski/Llama-3.3-70B-Instruct-GGUF:Q3_K_L", # 37.14 GB

]


## Automatic Excel sheet import for model list array

In [10]:
!pip install openpyxl -q


[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python -m pip install --upgrade pip


In [61]:
import pandas as pd

# Read the Excel file and select the "ollama-ggufs" sheet
file_path = "20241111 Benchmark Database.xlsx"  # Replace with your actual file path
sheet_name = "ollama-ggufs"

# Load the sheet into a DataFrame
df = pd.read_excel(file_path, sheet_name=sheet_name)

# Step 3a: Filter for models below 48GB
models_below_48GB = df[df["Model Size (GB)"] < 48].sort_values(by="Model Size (GB)", ascending=False)
count_below_48GB = models_below_48GB.shape[0]
print(f"Count of models below 48GB: {count_below_48GB}")
print("Models below 48GB (sorted by size descending):")
print(models_below_48GB[["Ollama Command", "Model Size (GB)"]])

# Step 3b: Filter for models above 48GB
models_above_48GB = df[df["Model Size (GB)"] >= 48].sort_values(by="Model Size (GB)", ascending=False)
count_above_48GB = models_above_48GB.shape[0]
print(f"\nCount of models above or equal to 48GB: {count_above_48GB}")
print("\nModels above 48GB (sorted by size descending):")
print(models_above_48GB[["Ollama Command", "Model Size (GB)"]])

# Select the model list based on the number of GPUs we're using
model_list = models_below_48GB["Ollama Command"].tolist()
# model_list = models_above_48GB["Ollama Command"].tolist()

Count of models below 48GB: 153
Models below 48GB (sorted by size descending):
                      Ollama Command  Model Size (GB)
5         llama3.3:70b-instruct-q4_1           44.000
6       llama3.3:70b-instruct-q4_K_M           43.000
4         llama3.3:70b-instruct-q4_0           40.000
7       llama3.3:70b-instruct-q4_K_S           40.000
146  mixtral:8x7b-instruct-v0.1-q6_K           37.000
..                               ...              ...
18         llama3.2:1b-instruct-q4_0            0.771
15       llama3.2:1b-instruct-q3_K_L            0.733
16       llama3.2:1b-instruct-q3_K_M            0.691
17       llama3.2:1b-instruct-q3_K_S            0.642
14         llama3.2:1b-instruct-q2_K            0.581

[153 rows x 2 columns]

Count of models above or equal to 48GB: 10

Models above 48GB (sorted by size descending):
                      Ollama Command  Model Size (GB)
0         llama3.3:70b-instruct-fp16            141.0
133  mixtral:8x7b-instruct-v0.1-fp16             

In [58]:
model_list

['llama3.3:70b-instruct-q4_1',
 'llama3.3:70b-instruct-q4_K_M',
 'llama3.3:70b-instruct-q4_0',
 'llama3.3:70b-instruct-q4_K_S',
 'mixtral:8x7b-instruct-v0.1-q6_K',
 'mixtral:8x7b-instruct-v0.1-q5_1',
 'qwen2.5:32b-instruct-q8_0',
 'llama3.3:70b-instruct-q3_K_M',
 'mixtral:8x7b-instruct-v0.1-q5_K_M',
 'mixtral:8x7b-instruct-v0.1-q5_0',
 'mixtral:8x7b-instruct-v0.1-q5_K_S',
 'llama3.3:70b-instruct-q3_K_S',
 'qwen2.5:14b-instruct-fp16',
 'mixtral:8x7b-instruct-v0.1-q4_1',
 'gemma2:27b-instruct-q8_0',
 'mixtral:8x7b-instruct-v0.1-q4_K_M',
 'qwen2.5:32b-instruct-q6_K',
 'mixtral:8x7b-instruct-v0.1-q4_0',
 'mixtral:8x7b-instruct-v0.1-q4_K_S',
 'llama3.3:70b-instruct-q2_K',
 'qwen2.5:32b-instruct-q5_1',
 'mixtral:8x7b-instruct-v0.1-q3_K_L',
 'qwen2.5:32b-instruct-q5_0',
 'qwen2.5:32b-instruct-q5_K_M',
 'qwen2.5:32b-instruct-q5_K_S',
 'gemma2:27b-instruct-q6_K',
 'qwen2.5:32b-instruct-q4_1',
 'gemma2:27b-instruct-q5_1',
 'mixtral:8x7b-instruct-v0.1-q3_K_M',
 'qwen2.5:32b-instruct-q4_K_M',
 'qw

## Using ollama python library
Does not pipe the status of the model pull. You have no idea what speed it's pulling the model.

In [7]:
import ollama

In [5]:
# Loop through the models and call ollama.pull
for model in model_list:
    print(f"Attempting to pull: {model}")  # Print before pulling
    try:
        ollama.pull(model)
        print(f"Successfully pulled: {model}")
    except Exception as e:
        print(f"Failed to pull: {model}. Error: {e}")


Attempting to pull: hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q8_0
Failed to pull: hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q8_0. Error: Server disconnected without sending a response.
Attempting to pull: hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q6_K_L
Failed to pull: hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q6_K_L. Error: [Errno 111] Connection refused
Attempting to pull: hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q6_K
Failed to pull: hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q6_K. Error: [Errno 111] Connection refused
Attempting to pull: hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q5_K_L
Failed to pull: hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q5_K_L. Error: [Errno 111] Connection refused


In [9]:
ollama.list()

{'models': []}

## Using cmd line commands
Pipe the status of the model pull so you can see how fast models are downloading.

In [ ]:
def pull_models(models):
    for model in models:
        print(f"Pulling model: {model}")
        !ollama pull {model}
        
failed_models = []

for model in model_list:
    try:
        # Debug: Current model being processed
        print(f"\n--- Processing model: {model} ---")

        # Pull the model
        print(f"Step 1: Pulling model: {model}")
        pull_models([model])
        print(f"Model {model} pulled successfully.\n")

    except Exception as e:
        # Debug: Handle any errors that occur
        print(f"An error occurred while processing model {model}: {e}")
        print("Skipping to the next model.\n")

        # Add the model to the failed list
        failed_models.append(model)

# Going for parallelization of cmd lines

This can be refined -- it's not reporting back...

In [ ]:
# not using pyqt anymore...
# !pip install pyqt5 -q

In [5]:
# import subprocess
# import threading
# import ipywidgets as widgets
# from IPython.display import display

# # Create GUI components
# output_widgets = {model: widgets.Textarea(value="Waiting...\n", layout=widgets.Layout(width="100%", height="100px")) for model in model_list}
# start_button = widgets.Button(description="Start Pulling")

# # Function to log the output for each model
# def log(model, message):
#     current_value = output_widgets[model].value
#     output_widgets[model].value = current_value + f"{message}\n"

# # Function to pull a single model
# def pull_model(model):
#     try:
#         log(model, "Starting pull...")

#         # Run the pull command in a separate subprocess
#         process = subprocess.Popen(
#             ["ollama", "pull", model],
#             stdout=subprocess.PIPE,
#             stderr=subprocess.PIPE,
#             text=True,
#             bufsize=1  # Ensure real-time output
#         )

#         # Read output and append all lines in real time
#         while True:
#             output = process.stdout.readline()
#             if output == "" and process.poll() is not None:
#                 break
#             if output:
#                 log(model, output.strip())

#         # Capture and log any errors
#         for line in process.stderr:
#             log(model, f"ERROR: {line.strip()}")

#         process.wait()  # Wait for the process to complete

#         if process.returncode == 0:
#             log(model, "Pull successful.")
#         else:
#             log(model, f"Pull failed with return code {process.returncode}.")
#     except Exception as e:
#         log(model, f"Error: {e}")

# # Function to start pulling models
# def start_pulling(_):
#     start_button.disabled = True  # Disable the button during execution

#     threads = []
#     for model in model_list:
#         thread = threading.Thread(target=pull_model, args=(model,))
#         threads.append(thread)
#         thread.start()

#     # Wait for all threads to complete
#     for thread in threads:
#         thread.join()

#     start_button.disabled = False  # Re-enable the button

# # Bind the start button to the function
# start_button.on_click(start_pulling)

# # Display the GUI
# subviews = [widgets.VBox([widgets.HTML(f"<b>{model}</b>"), output_widgets[model]]) for model in model_list]
# display(widgets.VBox([start_button] + subviews))


In [48]:
import subprocess
import threading
import ipywidgets as widgets
from IPython.display import display
import time

output_widgets = {model: widgets.Textarea(value="Waiting...\n", layout=widgets.Layout(width="100%", height="100px")) 
                 for model in model_list}
start_button = widgets.Button(description="Start Pulling")
status_label = widgets.HTML(value="Status: Ready")

def log(model, message):
    """Thread-safe logging to widgets"""
    current_value = output_widgets[model].value
    output_widgets[model].value = current_value + f"{message}\n"
    # Auto-scroll to bottom
    output_widgets[model].value = output_widgets[model].value

def pull_model(model):
    try:
        log(model, "Starting pull...")
        
        # Run the pull command in a separate subprocess
        process = subprocess.Popen(
            ["ollama", "pull", model],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            universal_newlines=True
        )
        
        # Track if we've seen any output
        had_output = False
        last_output_time = time.time()
        
        while True:
            # Check both stdout and stderr
            outputs = [
                (process.stdout.readline(), ""),
                (process.stderr.readline(), "ERROR: ")
            ]
            
            # Process any available output
            for output, prefix in outputs:
                if output:
                    had_output = True
                    last_output_time = time.time()
                    log(model, f"{prefix}{output.strip()}")
            
            # Check if process has finished
            if process.poll() is not None:
                break
                
            # If no output for 30 seconds, show a message
            if had_output and time.time() - last_output_time > 30:
                log(model, "Still pulling... (no new output for 30 seconds)")
                last_output_time = time.time()
            
            # Small sleep to prevent CPU spinning
            time.sleep(0.1)
        
        # Final status check
        if process.returncode == 0:
            log(model, "✅ Pull completed successfully.")
        else:
            log(model, f"❌ Pull failed with return code {process.returncode}.")
            
    except Exception as e:
        log(model, f"❌ Error: {str(e)}")
        raise e

def start_pulling(_):
    start_button.disabled = True
    status_label.value = "Status: Pulling models..."
    
    # Clear previous output
    for model in model_list:
        output_widgets[model].value = ""
    
    def run_pulls():
        try:
            threads = []
            for model in model_list:
                thread = threading.Thread(target=pull_model, args=(model,))
                threads.append(thread)
                thread.start()
            
            # Wait for all threads to complete
            for thread in threads:
                thread.join()
                
            status_label.value = "Status: All pulls completed"
        except Exception as e:
            status_label.value = f"Status: Error occurred - {str(e)}"
        finally:
            start_button.disabled = False
    
    # Run the pulls in a separate thread to keep UI responsive
    threading.Thread(target=run_pulls).start()

# Bind the start button to the function
start_button.on_click(start_pulling)

# Display the GUI
subviews = [widgets.VBox([widgets.HTML(f"<b>{model}</b>"), output_widgets[model]]) 
            for model in model_list]
display(widgets.VBox([start_button, status_label] + subviews))

Exception in thread Thread-189 (pull_model):
Traceback (most recent call last):
  File "/usr/lib/python3.11/threading.py", line 1045, in _bootstrap_inner
Exception in thread Thread-188 (pull_model):
Traceback (most recent call last):
  File "/usr/lib/python3.11/threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "/usr/lib/python3.11/threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_269/3469378465.py", line 70, in pull_model
    self.run()
  File "/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "/usr/lib/python3.11/threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_269/3469378465.py", line 70, in pull_model
  File "/tmp/ipykernel_269/3469378465.py", line 24, in pull_

In [ ]:
# trying a more compact reporting mechanism
import subprocess
import threading
import ipywidgets as widgets
from IPython.display import display
import time
import re

class CompactModelPuller:
    def __init__(self, model_list):
        self.model_list = model_list
        self.setup_ui()
        
    def setup_ui(self):
        # Create a row for each model with status and progress
        self.rows = {}
        table_rows = []
        
        for model in self.model_list:
            # Create status indicator (⏳ 🔄 ✅ ❌)
            status = widgets.HTML(value="⏳")
            
            # Create progress text
            progress = widgets.HTML(value="Waiting...", layout=widgets.Layout(width='200px'))
            
            # Create progress bar
            bar = widgets.FloatProgress(
                value=0, min=0, max=100,
                description='',
                bar_style='info',
                orientation='horizontal',
                layout=widgets.Layout(width='150px')
            )
            
            # Store widgets for this model
            self.rows[model] = {
                'status': status,
                'progress': progress,
                'bar': bar
            }
            
            # Create a row with model name (shortened), status, progress text, and bar
            short_name = model.split('/')[-1]  # Get just the model name part
            model_label = widgets.HTML(value=f"<code>{short_name}</code>", 
                                    layout=widgets.Layout(width='200px'))
            
            table_rows.append(widgets.HBox([
                model_label, status, progress, bar
            ], layout=widgets.Layout(padding='2px')))
        
        # Create main container
        self.container = widgets.VBox([
            widgets.HTML(value="<h3>Model Downloads</h3>"),
            widgets.Button(description='Start Pulling', 
                         on_click=self.start_pulling,
                         layout=widgets.Layout(width='150px')),
            widgets.VBox(table_rows, 
                        layout=widgets.Layout(border='1px solid #ddd', 
                                            padding='10px',
                                            margin='10px 0'))
        ])
        
    def update_model_status(self, model, status_emoji, progress_text, progress_value=None):
        """Update the status and progress for a model"""
        row = self.rows[model]
        row['status'].value = status_emoji
        row['progress'].value = progress_text
        if progress_value is not None:
            row['bar'].value = progress_value

    def parse_progress(self, line):
        """Parse progress information from ollama output"""
        progress = 0
        if 'downloading' in line.lower():
            match = re.search(r'(\d+)%', line)
            if match:
                progress = int(match.group(1))
        return progress

    def pull_model(self, model):
        try:
            self.update_model_status(model, "🔄", "Starting pull...", 0)
            
            process = subprocess.Popen(
                ["ollama", "pull", model],
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
                universal_newlines=True
            )
            
            while True:
                outputs = [
                    (process.stdout.readline(), ""),
                    (process.stderr.readline(), "ERROR: ")
                ]
                
                for output, prefix in outputs:
                    if output:
                        # Update progress bar if possible
                        progress = self.parse_progress(output)
                        if progress > 0:
                            self.update_model_status(model, "🔄", output.strip(), progress)
                        else:
                            self.update_model_status(model, "🔄", output.strip())
                
                if process.poll() is not None:
                    break
                    
                time.sleep(0.1)
            
            if process.returncode == 0:
                self.update_model_status(model, "✅", "Complete", 100)
            else:
                self.update_model_status(model, "❌", f"Failed (code {process.returncode})", 0)
                
        except Exception as e:
            self.update_model_status(model, "❌", f"Error: {str(e)}", 0)

    def start_pulling(self, _):
        # Disable the button
        self.container.children[1].disabled = True
        
        # Reset all status indicators
        for model in self.model_list:
            self.update_model_status(model, "⏳", "Waiting...", 0)
        
        def run_pulls():
            try:
                threads = []
                for model in self.model_list:
                    thread = threading.Thread(target=self.pull_model, args=(model,))
                    threads.append(thread)
                    thread.start()
                
                for thread in threads:
                    thread.join()
            finally:
                self.container.children[1].disabled = False
        
        threading.Thread(target=run_pulls).start()
        
    def display(self):
        display(self.container)

puller = CompactModelPuller(model_list)
puller.display()

# Comparing current models available in Ollama to model list

In [49]:
import ollama
ollama_data = ollama.list()

In [50]:
# Extract the list of model names
ollama_models_avail = [model['name'] for model in ollama_data['models']]


In [51]:
# ollama_models_avail

In [52]:
# Step 3: Compare model_list with Ollama's available models
if set(model_list) == set(ollama_models_avail):
    print("All models downloaded!")
else:
    missing_models = set(model_list) - set(ollama_models_avail)
    print(f"Missing models: {len(missing_models)}")
    print("Missing models:")
    for model in missing_models:
        print(f"- {model}")

Missing models: 14
Missing models:
- mixtral:8x7b-instruct-v0.1-q4_0
- llama3.3:70b-instruct-q4_K_M
- llama3.3:70b-instruct-q4_1
- gemma2:27b-instruct-q4_1
- gemma2:27b-instruct-q3_K_M
- qwen2.5:32b-instruct-q3_K_L
- qwen2.5:14b-instruct-q3_K_S
- gemma2:27b-instruct-q8_0
- mixtral:8x7b-instruct-v0.1-q4_1
- qwen2.5:32b-instruct
- qwen2.5:32b-instruct-q4_K_M
- llama3.3:70b-instruct-q4_K_S
- qwen2.5:14b-instruct-q8_0
- gemma2:9b-instruct-fp16


# Ollama Generate Check (SKIP)

In [5]:
# # Define the list of models to pull and test
# model_list = [
#     "hf.co/bartowski/Llama-3.2-3B-Instruct-GGUF:Q3_K_L",
#     "hf.co/bartowski/Llama-3.2-3B-Instruct-GGUF:Q4_K_S",
# ]

# # Test prompt
# test_prompt = "Why is the sky blue?"

# # Loop through the models and perform the actions
# for model in model_list:
#     print(f"Attempting to pull: {model}")  # Indicate which model is being pulled
#     try:
#         # Pull the model
#         ollama.pull(model)
#         print(f"Successfully pulled: {model}")

#         # Test the model with the prompt
#         print(f"Testing model: {model}")
#         response = ollama.generate(model=model, prompt=test_prompt)
#         print(f"Response from {model}: {response}")
#     except Exception as e:
#         print(f"Failed for model: {model}. Error: {e}")


# Checking that the server is up and running

In [18]:
import requests

def run_openai_api_chat_test(model, prompt, api_url="http://localhost:11434/v1/chat/completions"):
    """
    Runs an OpenAI-style chat completion API test against the specified model using Ollama's API.

    Args:
        model (str): The model to test (e.g., "llama-2-7b-chat").
        prompt (str): The input prompt to test with the model.
        api_url (str): The endpoint for the Ollama API (default: localhost:11434).

    Returns:
        dict: The response from the API.
    """
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},  # Optional system message
            {"role": "user", "content": prompt}
        ],
        "max_tokens": 100,
        "temperature": 0.7,
        "top_p": 1.0
    }
    try:
        response = requests.post(api_url, json=payload)
        response.raise_for_status()  # Raise an exception for HTTP errors
        return response.json()  # Return the response JSON
    except requests.exceptions.RequestException as e:
        raise RuntimeError(f"API test failed for model {model}: {e}")


In [19]:
##### regular compeltions API WORKS!!!
# Example prompt for testing
test_prompt = "Explain the importance of systems engineering."

# can use below as a subset if i dont want to test ALL of them.
# model_list = [
#     "hf.co/bartowski/Llama-3.2-3B-Instruct-GGUF:Q3_K_L",
#     "hf.co/bartowski/Llama-3.2-3B-Instruct-GGUF:Q4_K_S",
# ]

# Step 5: Workflow to pull, benchmark, and remove models
for model in model_list:
    try:
        # Step 3: Run the OpenAI-style Chat API test
        print(f"Step 4: Running OpenAI CHAT Completions API test for model: {model}")
        response2 = run_openai_api_chat_test(model, test_prompt)
        
        # PRINT GENERATED COMPLETION FROM CHAT COMPLETIONS API
        generated_text2 = response2.get("choices", [{}])[0].get("message", {}).get("content", "No output generated.")
        print(generated_text2)
    
    except Exception as e:
        # Debug: Handle any errors that occur
        print(f"An error occurred while processing model {model}: {e}")
        print("Skipping to the next model.\n")

        # Add the model to the failed list
        failed_models.append(model)


Step 4: Running OpenAI CHAT Completions API test for model: hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q8_0
Systems engineering is a crucial discipline that plays a vital role in ensuring the success of complex projects and systems across various industries, including aerospace, defense, healthcare, transportation, and more. The importance of systems engineering can be summarized as follows:

1. **Integrated Approach**: Systems engineering provides an integrated approach to designing, developing, testing, and deploying complex systems. It considers all aspects of a system, from requirements to operations, ensuring that they work together seamlessly.
2. **Risk Management**: By identifying and
Step 4: Running OpenAI CHAT Completions API test for model: hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q6_K_L
Systems engineering is a vital discipline that plays a crucial role in developing and managing complex systems, such as aerospace, defense, transportation, healthcare, energy, and

# Starting the eval

Remvoing below to try and help troubleshoot why it's always returning ##

generation_kwargs:
  temperature: 0.0
  max_tokens: 20
  until:
    - "</s>"
    - "\n"

ORIGINAL for SYSENGBENCH PAPER

doc_to_text: "Given the following question and four candidate answers (A, B, C and D), choose the best answer.
Question: {{question}}
A. {{choiceA}}
B. {{choiceB}}
C. {{choiceC}}
D. {{choiceD}}
Your response must consist solely of the single letter corresponding to the best answer, chosen from one of A, B, C or D.
Any response other than a single letter (A, B, C, or D) will be considered invalid.
Answer:"
doc_to_target: "{{answer}}"

NEW FOR QUANTIZATION PAPER

doc_to_text: "Given the following question and four candidate answers (A, B, C and D), choose the best answer.
Your response must consist solely of the single letter corresponding to the best answer, chosen from one of A, B, C or D.
Any response other than a single letter (A, B, C, or D) will be considered invalid.\n
Question: {{question}}\n
A. {{choiceA}}\n
B. {{choiceB}}\n
C. {{choiceC}}\n
D. {{choiceD}}\n
Answer:"
doc_to_target: "{{answer}}"

In [53]:
### updated version to match the MCQ setup
YAML_syseng_string = '''
task: sysengbench
dataset_path: rabell/SysEngBench
dataset_name: null
output_type: generate_until
training_split: null
validation_split: null
test_split: test
doc_to_text: "Given the following question and four candidate answers (A, B, C and D), choose the best answer.
Your response must consist solely of the single letter corresponding to the best answer, chosen from one of A, B, C or D.
Any response other than a single letter (A, B, C, or D) will be considered invalid.\n
Question: {{question}}\n
A. {{choiceA}}\n
B. {{choiceB}}\n
C. {{choiceC}}\n
D. {{choiceD}}\n
Answer:"
doc_to_target: "{{answer}}"

generation_kwargs:
  temperature: 0.0
  max_tokens: 20
  until:
    - "</s>"
    - "\n"

filter_list:
  - name: "strict-match"
    filter:
      - function: "regex"
        regex_pattern: "([ABCD])"
      - function: "take_first"
metric_list:
  - metric: exact_match
    aggregation: mean
    higher_is_better: true
    ignore_punctuation: true
    ignore_case: true
metadata:
  version: 1.0
dataset_kwargs:
  trust_remote_code: true
'''
with open('sysengbench.yaml', 'w') as f:
    f.write(YAML_syseng_string)

In [ ]:
import subprocess
failed_models = []

# Step 5: Workflow to pull, benchmark, and remove models
for model in model_list:
    try:
        # Debug: Current model being processed
        print(f"\n--- Processing model: {model} ---")

        # Step 3: Run the benchmark using the `!` method
        print(f"Step 3: Running benchmark for model: {model}")
        base_url = "http://localhost:11434/v1/chat/completions"  # Ensure Ollama server is running on this URL
        include_path = "./"
        tasks = "sysengbench"
        # limit = 5 # if want to use, need to add the --limit {limit} \ line below
        output_dir = "output/sysengbench/"
        log_samples = True
        batch_size = "auto"
        temperature = 0.0
        apply_chat_template = True

        # Construct the benchmark command dynamically
        log_samples_flag = "--log_samples" if log_samples else ""
        apply_template_flag = "--apply_chat_template" if apply_chat_template else ""

        # Optionally put the limit varable for debugging
        # --limit {limit} \

        command = f"""
        lm_eval \
            --model local-chat-completions \
            --model_args model='{model}',base_url='{base_url}',num_concurrent=1 \
            --include_path {include_path} \
            --tasks {tasks} \
            --output {output_dir} \
            {log_samples_flag} \
            --num_fewshot 0 \
            --batch_size {batch_size} \
            --gen_kwargs temperature={temperature} \
            {apply_template_flag}
        """
        
        # Run the command using the Jupyter `!` magic
        print(f"Running benchmark with command:\n{command}")
        !{command}

    except Exception as e:
        # Debug: Handle any errors that occur
        print(f"An error occurred while processing model {model}: {e}")
        print("Skipping to the next model.\n")

        # Add the model to the failed list
        failed_models.append(model)

# Eval with a timestamped log file (Verified)

In [62]:
# test a subset
# model_list = model_list[:2]


In [72]:
# model_list = ['mixtral:8x7b-instruct-v0.1-q4_0', 'gemma2:9b-instruct-q5_K_S', 'mistral:7b-instruct-v0.3-q5_0', 'gemma2:2b-instruct-q5_K_S', 'phi3.5:3.8b-mini-instruct-q3_K_L', 'llama3.2:3b-instruct-q5_K_M', 'gemma2:9b-instruct-q5_1', 'mistral:7b-instruct-v0.3-q5_K_M', 'llama3.2:1b-instruct-q4_K_S', 'gemma2:27b-instruct-q8_0', 'llama3.2:3b-instruct-q4_1', 'mistral:7b-instruct-v0.3-q3_K_L', 'phi3.5:3.8b-mini-instruct-q4_0', 'gemma2:2b-instruct-q4_1', 'llama3.3:70b-instruct-q4_K_S', 'llama3.2:1b-instruct-q3_K_S', 'qwen2.5:14b-instruct-q8_0', 'llama3.3:70b-instruct-q4_K_M', 'llama3.2:3b-instruct-q3_K_S', 'gemma2:2b-instruct-q4_K_M', 'llama3.2:3b-instruct-q5_1', 'llama3.2:1b-instruct-q6_K', 'qwen2.5:14b-instruct-q3_K_S', 'phi3.5:3.8b-mini-instruct-q3_K_S', 'qwen2.5:32b-instruct', 'gemma2:2b-instruct-q2_K', 'gemma2:2b-instruct-q5_K_M', 'gemma2:9b-instruct-q6_K', 'llama3.2:1b-instruct-q5_0', 'phi3.5:3.8b-mini-instruct-q4_K_M', 'llama3.3:70b-instruct-q4_1', 'gemma2:2b-instruct-q3_K_L', 'mixtral:8x7b-instruct-v0.1-q4_1', 'llama3.2:1b-instruct-q5_K_S', 'qwen2.5:32b-instruct-q4_K_M', 'llama3.2:1b-instruct-q3_K_L', 'llama3.2:3b-instruct-q3_K_M', 'llama3.2:3b-instruct-q5_0', 'qwen2.5:14b-instruct-q3_K_M', 'phi3.5:3.8b-mini-instruct-q5_0', 'gemma2:2b-instruct-fp16', 'qwen2.5:14b-instruct-q3_K_L', 'gemma2:9b-instruct-fp16', 'qwen2.5:14b-instruct-q2_K', 'llama3.2:3b-instruct-q5_K_S', 'gemma2:27b-instruct-q4_1', 'mistral:7b-instruct-v0.3-q4_K_S', 'phi3.5:3.8b-mini-instruct-fp16', 'mistral:7b-instruct-v0.3-q4_K_M', 'phi3.5:3.8b-mini-instruct-q5_1', 'gemma2:9b-instruct-q4_1', 'gemma2:9b-instruct-q4_0', 'mistral:7b-instruct-v0.3-q6_K', 'gemma2:9b-instruct-q3_K_M', 'phi3.5:3.8b-mini-instruct-q4_1', 'mistral:7b-instruct-v0.3-q4_1', 'qwen2.5:14b-instruct-q4_K_S', 'gemma2:9b-instruct-q4_K_S', 'gemma2:9b-instruct-q2_K', 'phi3.5:3.8b-mini-instruct-q2_K', 'llama3.2:3b-instruct-q4_K_S', 'llama3.2:1b-instruct-fp16', 'gemma2:2b-instruct-q4_K_S', 'gemma2:2b-instruct-q5_1', 'qwen2.5:32b-instruct-q3_K_L', 'llama3.2:3b-instruct-q4_K_M', 'gemma2:2b-instruct-q3_K_S', 'gemma2:2b-instruct-q3_K_M', 'phi3.5:3.8b-mini-instruct-q5_K_M', 'llama3.2:1b-instruct-q2_K', 'phi3.5:3.8b-mini-instruct-q6_K', 'gemma2:9b-instruct-q3_K_S', 'gemma2:2b-instruct-q4_0', 'llama3.2:3b-instruct-q4_0', 'gemma2:2b-instruct-q8_0', 'phi3.5:3.8b-mini-instruct-q4_K_S', 'gemma2:2b-instruct-q6_K', 'llama3.2:1b-instruct-q3_K_M', 'phi3.5:3.8b-mini-instruct-q5_K_S', 'mistral:7b-instruct-v0.3-q5_1', 'mistral:7b-instruct-v0.3-q5_K_S', 'phi3.5:3.8b-mini-instruct-q8_0', 'gemma2:2b-instruct-q5_0', 'gemma2:9b-instruct-q4_K_M', 'qwen2.5:14b-instruct-q4_0', 'llama3.2:3b-instruct-fp16', 'gemma2:27b-instruct-q3_K_M', 'llama3.2:1b-instruct-q4_K_M', 'mistral:7b-instruct-v0.3-q3_K_M', 'mistral:7b-instruct-v0.3-q3_K_S', 'mistral:7b-instruct-v0.3-q2_K', 'llama3.2:1b-instruct-q5_K_M', 'llama3.2:1b-instruct-q4_1', 'llama3.2:1b-instruct-q4_0', 'llama3.2:3b-instruct-q3_K_L', 'llama3.2:3b-instruct-q6_K', 'mistral:7b-instruct-v0.3-q8_0', 'gemma2:9b-instruct-q3_K_L', 'phi3.5:3.8b-mini-instruct-q3_K_M', 'llama3.2:3b-instruct-q2_K', 'llama3.2:1b-instruct-q5_1', 'mistral:7b-instruct-v0.3-q4_0', 'gemma2:9b-instruct-q5_K_M', 'gemma2:9b-instruct-q5_0']
# model_list = ['mixtral:8x7b-instruct-v0.1-q4_0', 'mistral:7b-instruct-v0.3-q5_K_S', 'phi3.5:3.8b-mini-instruct-q8_0', 'gemma2:2b-instruct-q5_0', 'gemma2:9b-instruct-q4_K_M', 'qwen2.5:14b-instruct-q4_0', 'llama3.2:3b-instruct-fp16', 'llama3.3:70b-instruct-q4_1', 'gemma2:27b-instruct-q3_K_M', 'llama3.2:1b-instruct-q4_K_M', 'gemma2:27b-instruct-q8_0', 'mixtral:8x7b-instruct-v0.1-q4_1', 'qwen2.5:32b-instruct-q4_K_M', 'llama3.2:3b-instruct-q4_K_S', 'llama3.2:1b-instruct-fp16', 'mistral:7b-instruct-v0.3-q3_K_M', 'gemma2:2b-instruct-q4_K_S', 'gemma2:2b-instruct-q5_1', 'mistral:7b-instruct-v0.3-q3_K_S', 'mistral:7b-instruct-v0.3-q2_K', 'llama3.2:1b-instruct-q5_K_M', 'qwen2.5:32b-instruct-q3_K_L', 'llama3.2:3b-instruct-q4_K_M', 'gemma2:2b-instruct-q3_K_S', 'gemma2:2b-instruct-q3_K_M', 'phi3.5:3.8b-mini-instruct-q5_K_M', 'llama3.2:1b-instruct-q2_K', 'phi3.5:3.8b-mini-instruct-q6_K', 'gemma2:9b-instruct-q3_K_S', 'gemma2:2b-instruct-q4_0', 'llama3.3:70b-instruct-q4_K_S', 'llama3.2:1b-instruct-q4_1', 'llama3.2:1b-instruct-q4_0', 'llama3.2:3b-instruct-q3_K_L', 'llama3.2:3b-instruct-q6_K', 'llama3.2:3b-instruct-q4_0', 'qwen2.5:14b-instruct-q8_0', 'gemma2:9b-instruct-fp16', 'llama3.3:70b-instruct-q4_K_M', 'mistral:7b-instruct-v0.3-q8_0', 'gemma2:9b-instruct-q3_K_L', 'gemma2:27b-instruct-q4_1', 'phi3.5:3.8b-mini-instruct-q3_K_M', 'gemma2:2b-instruct-q8_0', 'phi3.5:3.8b-mini-instruct-q4_K_S', 'gemma2:2b-instruct-q6_K', 'qwen2.5:14b-instruct-q3_K_S', 'llama3.2:1b-instruct-q3_K_M', 'llama3.2:3b-instruct-q2_K', 'llama3.2:1b-instruct-q5_1', 'phi3.5:3.8b-mini-instruct-q5_K_S', 'qwen2.5:32b-instruct', 'mistral:7b-instruct-v0.3-q5_1', 'mistral:7b-instruct-v0.3-q4_0', 'gemma2:9b-instruct-q5_K_M', 'gemma2:9b-instruct-q5_0']
model_list = ['mixtral:8x7b-instruct-v0.1-q4_0', 'llama3.3:70b-instruct-q4_K_M', 'llama3.3:70b-instruct-q4_1', 'gemma2:27b-instruct-q3_K_M', 'gemma2:27b-instruct-q4_1', 'qwen2.5:32b-instruct-q3_K_L', 'qwen2.5:14b-instruct-q3_K_S', 'gemma2:27b-instruct-q8_0', 'mixtral:8x7b-instruct-v0.1-q4_1', 'qwen2.5:32b-instruct-q4_K_M', 'qwen2.5:32b-instruct', 'llama3.3:70b-instruct-q4_K_S', 'qwen2.5:14b-instruct-q8_0', 'gemma2:9b-instruct-fp16']

In [ ]:
import subprocess
from datetime import datetime

# Generate a unique log file name with a timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
progress_file = f"progress_log_{timestamp}.txt"  # File to log progress

failed_models = []
total_models = len(model_list)  # Total number of models to process

# Create the log file with a header
with open(progress_file, "w") as file:
    file.write(f"Progress Log - Batch Run {timestamp}\n")
    file.write("=================================\n")

# Step 5: Workflow to pull, benchmark, and remove models
for idx, model in enumerate(model_list, start=1):  # Enumerate for progress tracking
    try:
        # Get the current timestamp for processing
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        # Debug: Current model being processed
        print(f"\n--- Processing model: {model} ({idx}/{total_models}) at {current_time} ---")
        
        # Log the current model and progress to the file with timestamp
        with open(progress_file, "a") as file:
            file.write(f"[{current_time}] Processing model: {model} ({idx}/{total_models})\n")
        
        # Step 3: Run the benchmark using subprocess
        print(f"Step 3: Running benchmark for model: {model}")
        base_url = "http://localhost:11434/v1/chat/completions"  # Ensure Ollama server is running on this URL
        include_path = "./"
        tasks = "sysengbench"
        output_dir = "output/sysengbench/"
        log_samples = True
        batch_size = "auto"
        temperature = 0.0
        apply_chat_template = True

        # Construct the benchmark command dynamically
        log_samples_flag = "--log_samples" if log_samples else ""
        apply_template_flag = "--apply_chat_template" if apply_chat_template else ""

        command = f"""
        lm_eval \
            --model local-chat-completions \
            --model_args model='{model}',base_url='{base_url}',num_concurrent=1 \
            --include_path {include_path} \
            --tasks {tasks} \
            --output {output_dir} \
            {log_samples_flag} \
            --num_fewshot 0 \
            --batch_size {batch_size} \
            --gen_kwargs temperature={temperature} \
            {apply_template_flag}
        """

        # Run the command using subprocess
        print(f"Running benchmark with command:\n{command}")
        subprocess.run(command, shell=True, check=True)

    except Exception as e:
        # Get the current timestamp for the error log
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        # Debug: Handle any errors that occur
        print(f"An error occurred while processing model {model} at {current_time}: {e}")
        print("Skipping to the next model.\n")

        # Log the failure to the file with timestamp
        with open(progress_file, "a") as file:
            file.write(f"[{current_time}] Failed to process model: {model}\n")

        # Add the model to the failed list
        failed_models.append(model)


# re-running the missing Models

In [71]:
import os

# Path to the directory containing output folders
output_dir = "output/sysengbench/"

# Step 1: List all folder names in the output directory
try:
    folder_names = [name for name in os.listdir(output_dir) if os.path.isdir(os.path.join(output_dir, name))]
    print("Folders found in output directory:", folder_names)
except FileNotFoundError:
    print(f"Directory {output_dir} not found.")
    folder_names = []

# Step 2: Transform folder names to match the model list format
transformed_folder_names = [name.replace("__", ":") for name in folder_names]

# Step 3: Compare transformed folder names to model_list
ran_models = set(transformed_folder_names)
remaining_models = set(model_list) - ran_models

# Step 4: Display results
print("\nModels that have been run:")
for model in ran_models:
    print(f"- {model}")

print("\nModels that still need to be run:")
for model in remaining_models:
    print(f"- {model}")

# Step 5: Generate a ready-to-copy array for remaining models
remaining_models_array = list(remaining_models)
print("\nCopy and paste the following into your script:")
print(f"model_list = {remaining_models_array}")


Folders found in output directory: ['gemma2__9b-instruct-q5_0', 'gemma2__9b-instruct-q5_K_M', 'mistral__7b-instruct-v0.3-q4_0', 'mistral__7b-instruct-v0.3-q5_1', 'phi3.5__3.8b-mini-instruct-q5_K_S', 'llama3.2__1b-instruct-q5_1', 'llama3.2__3b-instruct-q2_K', 'llama3.2__1b-instruct-q3_K_M', 'gemma2__2b-instruct-q6_K', 'phi3.5__3.8b-mini-instruct-q4_K_S', 'gemma2__2b-instruct-q8_0', 'phi3.5__3.8b-mini-instruct-q3_K_M', 'gemma2__9b-instruct-q3_K_L', 'mistral__7b-instruct-v0.3-q8_0', 'llama3.2__3b-instruct-q4_0', 'llama3.2__3b-instruct-q6_K', 'llama3.2__3b-instruct-q3_K_L', 'llama3.2__1b-instruct-q4_0', 'llama3.2__1b-instruct-q4_1', 'gemma2__2b-instruct-q4_0', 'gemma2__9b-instruct-q3_K_S', 'phi3.5__3.8b-mini-instruct-q6_K', 'llama3.2__1b-instruct-q2_K', 'phi3.5__3.8b-mini-instruct-q5_K_M', 'gemma2__2b-instruct-q3_K_M', 'gemma2__2b-instruct-q3_K_S', 'llama3.2__3b-instruct-q4_K_M', 'llama3.2__1b-instruct-q5_K_M', 'mistral__7b-instruct-v0.3-q2_K', 'mistral__7b-instruct-v0.3-q3_K_S', 'gemma2__